In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import tensorflow as tf

from pathlib import Path
from sysconfig import get_paths
import ctypes
import os

site_packages = Path(get_paths()['purelib'])
cuda_lib_dirs = [
    site_packages / 'nvidia' / 'cuda_runtime' / 'lib',
    site_packages / 'nvidia' / 'cuda_nvrtc' / 'lib',
    site_packages / 'nvidia' / 'cublas' / 'lib',
    site_packages / 'nvidia' / 'cufft' / 'lib',
    site_packages / 'nvidia' / 'curand' / 'lib',
    site_packages / 'nvidia' / 'cusolver' / 'lib',
    site_packages / 'nvidia' / 'cusparse' / 'lib',
    site_packages / 'nvidia' / 'cudnn' / 'lib',
    site_packages / 'nvidia' / 'cuda_cupti' / 'lib',
    site_packages / 'nvidia' / 'nccl' / 'lib',
    site_packages / 'nvidia' / 'nvjitlink' / 'lib',
]

loaded_cuda_libs = 0
for lib_dir in cuda_lib_dirs:
    if lib_dir.exists():
        for lib_path in sorted(lib_dir.glob('*.so*')):
            ctypes.CDLL(str(lib_path), mode=ctypes.RTLD_GLOBAL)
            loaded_cuda_libs += 1

existing_ld = os.environ.get('LD_LIBRARY_PATH', '')
cuda_ld_paths = [str(path) for path in cuda_lib_dirs if path.exists()]
if existing_ld:
    cuda_ld_paths.append(existing_ld)
os.environ['LD_LIBRARY_PATH'] = ':'.join(cuda_ld_paths)
print(f'Librerías CUDA cargadas: {loaded_cuda_libs}')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow:', tf.__version__)
print('GPUs detectadas:', gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ Crecimiento de memoria habilitado.")
    except RuntimeError as e:
        print(e)

In [ ]:
# ============================================================
# 2. ALGORITMO GREY WOLF OPTIMIZER (GWO)
# ============================================================
class GreyWolfOptimizer:
    """
    Grey Wolf Optimizer para optimizar pesos de red neuronal
    """
    def __init__(self, n_wolves=30, max_iter=100, dim=0):
        self.n_wolves = n_wolves  # Tamaño de la manada
        self.max_iter = max_iter  # Máximo de iteraciones
        self.dim = dim  # Dimensión del problema (número de pesos)
        
        # Posiciones de los lobos
        self.positions = np.random.uniform(-1, 1, (n_wolves, dim))
        self.fitness = np.zeros(n_wolves)
        
        # Mejores posiciones (Alpha, Beta, Delta)
        self.alpha_pos = np.zeros(dim)
        self.alpha_score = float('inf')
        self.beta_pos = np.zeros(dim)
        self.beta_score = float('inf')
        self.delta_pos = np.zeros(dim)
        self.delta_score = float('inf')
        
        # Historial para animación
        self.history = []
        
    def evaluate_fitness(self, positions, X, y, model_architecture):
        """Evalúa la fitness de cada lobo (pesos de la red)"""
        fitness_values = []
        
        for pos in positions:
            # Asignar pesos al modelo
            model = self.create_model_from_weights(pos, model_architecture)
            
            # Predecir
            y_pred = model.predict(X, verbose=0)
            y_pred_binary = (y_pred > 0.5).astype(int)
            
            # Calcular fitness (1 - accuracy para minimizar)
            accuracy = accuracy_score(y, y_pred_binary)
            fitness = 1 - accuracy
            
            fitness_values.append(fitness)
        
        return np.array(fitness_values)
    
    def create_model_from_weights(self, weights, architecture):
        """Crea un modelo Keras con pesos específicos"""
        model = Sequential()
        
        idx = 0
        for layer_config in architecture:
            if layer_config['type'] == 'dense':
                layer = Dense(
                    layer_config['units'],
                    activation=layer_config.get('activation', 'relu'),
                    input_dim=layer_config.get('input_dim', None)
                )
                model.add(layer)
                
                # Asignar pesos
                n_weights = layer_config['units'] * layer_config.get('input_dim', 1)
                w = weights[idx:idx + n_weights].reshape(
                    layer_config.get('input_dim', 1), 
                    layer_config['units']
                )
                idx += n_weights
                
                b = weights[idx:idx + layer_config['units']]
                idx += layer_config['units']
                
                layer.set_weights([w, b])
        
        # Capa de salida
        model.add(Dense(1, activation='sigmoid'))
        w_out = weights[idx:idx + architecture[-1]['units']].reshape(-1, 1)
        idx += architecture[-1]['units']
        b_out = weights[idx:idx + 1]
        model.layers[-1].set_weights([w_out, b_out])
        
        return model
    
    def optimize(self, X, y, model_architecture):
        """Ejecuta el algoritmo GWO"""
        print(f"Iniciando GWO con {self.n_wolves} lobos y {self.max_iter} iteraciones...")
        
        for iteration in range(self.max_iter):
            # Actualizar parámetro a (decrece de 2 a 0)
            a = 2 - iteration * (2 / self.max_iter)
            
            # Evaluar fitness
            self.fitness = self.evaluate_fitness(
                self.positions, X, y, model_architecture
            )
            
            # Actualizar Alpha, Beta, Delta
            for i in range(self.n_wolves):
                if self.fitness[i] < self.alpha_score:
                    self.delta_score = self.beta_score
                    self.delta_pos = self.beta_pos.copy()
                    self.beta_score = self.alpha_score
                    self.beta_pos = self.alpha_pos.copy()
                    self.alpha_score = self.fitness[i]
                    self.alpha_pos = self.positions[i].copy()
                elif self.fitness[i] < self.beta_score:
                    self.delta_score = self.beta_score
                    self.delta_pos = self.beta_pos.copy()
                    self.beta_score = self.fitness[i]
                    self.beta_pos = self.positions[i].copy()
                elif self.fitness[i] < self.delta_score:
                    self.delta_score = self.fitness[i]
                    self.delta_pos = self.positions[i].copy()
            
            # Guardar historial para animación
            if iteration % 5 == 0:
                self.history.append({
                    'iteration': iteration,
                    'positions': self.positions.copy(),
                    'alpha_pos': self.alpha_pos.copy(),
                    'alpha_score': self.alpha_score
                })
            
            # Actualizar posiciones de los lobos
            for i in range(self.n_wolves):
                for j in range(self.dim):
                    r1 = np.random.random()
                    r2 = np.random.random()
                    
                    A1 = 2 * a * r1 - a
                    C1 = 2 * r2
                    
                    D_alpha = abs(C1 * self.alpha_pos[j] - self.positions[i, j])
                    X1 = self.alpha_pos[j] - A1 * D_alpha
                    
                    r1 = np.random.random()
                    r2 = np.random.random()
                    
                    A2 = 2 * a * r1 - a
                    C2 = 2 * r2
                    
                    D_beta = abs(C2 * self.beta_pos[j] - self.positions[i, j])
                    X2 = self.beta_pos[j] - A2 * D_beta
                    
                    r1 = np.random.random()
                    r2 = np.random.random()
                    
                    A3 = 2 * a * r1 - a
                    C3 = 2 * r2
                    
                    D_delta = abs(C3 * self.delta_pos[j] - self.positions[i, j])
                    X3 = self.delta_pos[j] - A3 * D_delta
                    
                    self.positions[i, j] = (X1 + X2 + X3) / 3
            
            # Mostrar progreso
            if (iteration + 1) % 10 == 0:
                print(f"Iteración {iteration + 1}/{self.max_iter} - "
                      f"Mejor fitness: {self.alpha_score:.4f} - "
                      f"Accuracy: {(1 - self.alpha_score) * 100:.2f}%")
        
        return self.alpha_pos, self.alpha_score

In [ ]:
# ============================================================
# 3. DEFINIR ARQUITECTURA DEL MODELO
# ============================================================
model_architecture = [
    {'type': 'dense', 'units': 32, 'activation': 'relu', 'input_dim': 23},
    {'type': 'dense', 'units': 16, 'activation': 'relu', 'input_dim': 32}
]

# Calcular dimensión total (número de pesos)
total_dim = 0
for layer in model_architecture:
    total_dim += layer['units'] * layer.get('input_dim', 1)
    total_dim += layer['units']
total_dim += 1 * model_architecture[-1]['units'] + 1  # Capa de salida

print(f"Dimensión total del problema: {total_dim} pesos a optimizar")

In [ ]:
# ============================================================
# 4. ENTRENAR CON GWO USANDO GPU
# ============================================================
# Usar un subset de datos para demostración (GWO es computacionalmente costoso)
X_train_subset = X_train_scaled[:1000]
y_train_subset = y_train[:1000]

gwo = GreyWolfOptimizer(
    n_wolves=20,
    max_iter=50,
    dim=total_dim
)

best_weights, best_fitness = gwo.optimize(
    X_train_subset, y_train_subset, model_architecture
)

print(f"\n✅ Optimización completada!")
print(f"Mejor fitness: {best_fitness:.4f}")
print(f"Accuracy en entrenamiento: {(1 - best_fitness) * 100:.2f}%")

In [ ]:
# ============================================================
# 5. EVALUAR MODELO HÍBRIDO
# ============================================================
# Crear modelo final con mejores pesos
final_model = gwo.create_model_from_weights(best_weights, model_architecture)

# Predecir en test set
y_pred_prob = final_model.predict(X_test_scaled, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\n📊 Resultados del Modelo Híbrido GWO-MLP:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")